<a href="https://colab.research.google.com/github/03sarath/adk-agent-engine/blob/main/adk_agent_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ADK Deploy to Vertex AI Agent Engine

Agent Engine is a fully managed Google Cloud service enabling developers to deploy, manage, and scale AI agents in production. Agent Engine handles the infrastructure to scale agents in production so you can focus on creating intelligent and impactful applications.

This guide provides a step-by-step walkthrough for deploying an agent from your local environment.

**Note**: This notebook is optimized for Google Colab.


## Prerequisites

Before you begin, ensure you have the following:

1. **Google Cloud Project**: A Google Cloud project with the Vertex AI API enabled.
2. **Authenticated gcloud CLI**: You need to be authenticated with Google Cloud.
3. **Google Cloud Storage (GCS) Bucket**: Agent Engine requires a GCS bucket to stage your agent's code and dependencies for deployment.
4. **Python Environment**: Colab provides Python 3.9+ which is compatible.

**Important**: Make sure to enable the Vertex AI API in your Google Cloud project.

In [1]:
# Install required packages
!pip install google-cloud-aiplatform[adk,agent_engines]>=1.111
!pip install google-cloud-storage

In [2]:
# Authenticate with Google Cloud
from google.colab import auth
auth.authenticate_user()


In [3]:
# Install and initialize gcloud
!gcloud auth application-default login --quiet

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fapplicationdefaultauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=CxKoHnGE7OCIjJDkFCsK8exzNVEZLV&prompt=consent&token_usage=remote&access_type=offline&code_challenge=ffiDUPCoSkAXNYgg664WxEkS1xjWdnzUt1KgHBxsXCk&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AVMBsJjBgwW6he4b6HqXQMSuvMrwsYmrFP1vuG7TX9WJt-bZMd1TzxOz5_TWHdYT9H2bjA

Credentials saved to file: [/content/.config/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).
Ca

In [4]:
# Import required libraries
import datetime
from zoneinfo import ZoneInfo
import vertexai
from vertexai import agent_engines
from google.adk.agents import Agent
from google.genai import types
import asyncio

In [13]:
# Configuration - UPDATE THESE VALUES
PROJECT_ID = "mcp-test-471607"  # Replace with your project ID
LOCATION = "us-central1"  # or your preferred region
STAGING_BUCKET = "gs://aiagnetbkt"  # Replace with your bucket name

print(f"Project ID: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Staging Bucket: {STAGING_BUCKET}")

# Initialize Vertex AI
vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET,
)

Project ID: mcp-test-471607
Location: us-central1
Staging Bucket: gs://aiagnetbkt


## Step 1: Define Your Agent

First, define your agent. You can use the sample agent below, which has two tools (to get weather or retrieve the time in a specified city):

In [6]:
# Define the agent with tools
def get_weather(city: str) -> dict:
    """Retrieves the current weather report for a specified city.

    Args:
        city (str): The name of the city for which to retrieve the weather report.

    Returns:
        dict: status and result or error msg.
    """
    if city.lower() == "new york":
        return {
            "status": "success",
            "report": (
                "The weather in New York is sunny with a temperature of 25 degrees"
                " Celsius (77 degrees Fahrenheit)."
            ),
        }
    else:
        return {
            "status": "error",
            "error_message": f"Weather information for '{city}' is not available.",
        }


def get_current_time(city: str) -> dict:
    """Returns the current time in a specified city.

    Args:
        city (str): The name of the city for which to retrieve the current time.

    Returns:
        dict: status and result or error msg.
    """

    if city.lower() == "new york":
        tz_identifier = "America/New_York"
    else:
        return {
            "status": "error",
            "error_message": (
                f"Sorry, I don't have timezone information for {city}."
            ),
        }

    tz = ZoneInfo(tz_identifier)
    now = datetime.datetime.now(tz)
    report = (
        f'The current time in {city} is {now.strftime("%Y-%m-%d %H:%M:%S %Z%z")}'
    )
    return {"status": "success", "report": report}


# Create the agent
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.0-flash",
    description=(
        "Agent to answer questions about the time and weather in a city."
    ),
    instruction=(
        "You are a helpful agent who can answer user questions about the time and weather in a city."
    ),
    tools=[get_weather, get_current_time],
)

print("Agent created successfully!")

Agent created successfully!


## Step 2: Prepare the Agent for Deployment

To make your agent compatible with Agent Engine, you need to wrap it in an AdkApp object.

In [7]:
# Wrap the agent in an AdkApp object
app = agent_engines.AdkApp(
    agent=root_agent,
    enable_tracing=True,
)

print("AdkApp created successfully!")

AdkApp created successfully!


## Step 3: Test Your Agent Locally (Optional)

Before deploying, you can test your agent's behavior locally.

The `async_stream_query` method returns a stream of events that represent the agent's execution trace.

In [8]:
# Test the agent locally
async def test_agent():
    # Create a local session to maintain conversation history
    session = await app.async_create_session(user_id="u_123")
    print("Session created:", session)

    # Send a query to the agent
    events = []
    async for event in app.async_stream_query(
        user_id="u_123",
        session_id=session.id,
        message="whats the weather in new york",
    ):
        events.append(event)

    # Display the results
    print("\n--- Full Event Stream ---")
    for event in events:
        print(event)

    # Extract final text response
    final_text_responses = [
        e for e in events
        if e.get("content", {}).get("parts", [{}])[0].get("text")
        and not e.get("content", {}).get("parts", [{}])[0].get("function_call")
    ]
    if final_text_responses:
        print("\n--- Final Response ---")
        print(final_text_responses[0]["content"]["parts"][0]["text"])

# Run the test
await test_agent()

/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Session created: id='763e31d9-ec1f-45e0-92d3-a59887586435' app_name='default-app-name' user_id='u_123' state={} events=[] last_update_time=1757439959.9708078


/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)



--- Full Event Stream ---
{'content': {'parts': [{'function_call': {'id': 'adk-f7b2651c-6a4e-4110-a36f-cf91a7b39dca', 'args': {'city': 'new york'}, 'name': 'get_weather'}}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 6, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 6}], 'prompt_token_count': 180, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 180}], 'total_token_count': 186, 'traffic_type': 'ON_DEMAND'}, 'invocation_id': 'e-0a9d8b7b-15b9-4bd8-a00e-3f5a063eeb9f', 'author': 'weather_time_agent', 'actions': {'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}}, 'long_running_tool_ids': [], 'id': '36e1ce5a-32f7-449e-8672-bebd4be80231', 'timestamp': 1757439959.973546}
{'content': {'parts': [{'function_response': {'id': 'adk-f7b2651c-6a4e-4110-a36f-cf91a7b39dca', 'name': 'get_weather', 'response': {'status': 'success', 'report': 'The weather in New York is sunny with a temperature of 25 degrees

## Step 4: Deploy to Agent Engine

Once you are satisfied with your agent's local behavior, you can deploy it using the Python SDK.

This process packages your code, builds it into a container, and deploys it to the managed Agent Engine service. This can take several minutes.

In [14]:
# Deploy the agent to Agent Engine
def deploy_agent():
    try:
        print("Starting deployment...")
        remote_app = agent_engines.create(
            agent_engine=app,  # 'app' is your AdkApp instance
            requirements=[
                "google-cloud-aiplatform[adk,agent_engines]",
            ]
        )
        print(f"✅ Deployed successfully!")
        print(f"Resource name: {remote_app.resource_name}")
        return remote_app
    except Exception as e:
        print(f"❌ Deployment failed: {e}")
        return None

# Deploy the agent
remote_app = deploy_agent()

INFO:vertexai.agent_engines:Identified the following requirements: {'google-cloud-aiplatform': '1.111.0', 'pydantic': '2.11.7', 'cloudpickle': '3.1.1'}
INFO:vertexai.agent_engines:The following requirements are appended: {'cloudpickle==3.1.1', 'pydantic==2.11.7'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[adk,agent_engines]', 'cloudpickle==3.1.1', 'pydantic==2.11.7']


Starting deployment...


/usr/local/lib/python3.12/dist-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
INFO:vertexai.agent_engines:Using bucket aiagnetbkt
INFO:vertexai.agent_engines:Wrote to gs://aiagnetbkt/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://aiagnetbkt/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://aiagnetbkt/agent_engine/dependencies.tar.gz

For further information visit https://errors.pydantic.dev/2.11/u/class-not-fully-defined
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing 

✅ Deployed successfully!
Resource name: projects/887819871499/locations/us-central1/reasoningEngines/2186150173413998592


## Create Remote Session

In [20]:
# Create a remote session using async methods
async def create_remote_session():
    if remote_app:
        try:
            print(f"Deployed agent resource name: {remote_app.resource_name}")

            # Method 1: Try with remote_app (AgentEngine)
            try:
                print("Trying async_create_session with remote_app...")
                remote_session = await remote_app.async_create_session(user_id="u_456")
                print("✅ Remote session created with remote_app:", remote_session)
                return remote_session, remote_app
            except Exception as e1:
                print(f"❌ Remote app method failed: {e1}")

            # Method 2: Try with original app (AdkApp)
            try:
                print("Trying async_create_session with original app...")
                remote_session = await app.async_create_session(user_id="u_456")
                print("✅ Remote session created with original app:", remote_session)
                return remote_session, app
            except Exception as e2:
                print(f"❌ Original app method failed: {e2}")

            return None, None

        except Exception as e:
            print(f"❌ Failed to create remote session: {e}")
            return None, None
    else:
        print("No deployed app available")
        return None, None

# Create the session (async)
remote_session, adk_app = await create_remote_session()

Deployed agent resource name: projects/887819871499/locations/us-central1/reasoningEngines/2186150173413998592
Trying async_create_session with remote_app...
✅ Remote session created with remote_app: {'appName': 'default-app-name', 'events': [], 'state': {}, 'id': '8453016257283751936', 'lastUpdateTime': 1757441522.767476, 'userId': 'u_456'}


## Query Function (Async)

In [21]:
# Send queries to the remote agent using async methods
async def query_remote_agent():
    if adk_app and remote_session:
        try:
            print("Sending query to remote agent...")
            events = []
            async for event in adk_app.async_stream_query(
                user_id="u_456",
                session_id=remote_session["id"],
                message="whats the weather in new york",
            ):
                events.append(event)
                print("Event:", event)

            # Extract final response
            final_text_responses = [
                e for e in events
                if e.get("content", {}).get("parts", [{}])[0].get("text")
                and not e.get("content", {}).get("parts", [{}])[0].get("function_call")
            ]
            if final_text_responses:
                print("\n--- Final Response ---")
                print(final_text_responses[0]["content"]["parts"][0]["text"])

        except Exception as e:
            print(f"Query failed: {e}")
    else:
        print("No remote app or session available")

# Send the query (async)
await query_remote_agent()

Sending query to remote agent...
Event: {'content': {'parts': [{'function_call': {'id': 'adk-411dfb07-a0fb-4357-b72c-652d7a86fff0', 'args': {'city': 'new york'}, 'name': 'get_weather'}}], 'role': 'model'}, 'finish_reason': 'STOP', 'usage_metadata': {'candidates_token_count': 6, 'candidates_tokens_details': [{'modality': 'TEXT', 'token_count': 6}], 'prompt_token_count': 180, 'prompt_tokens_details': [{'modality': 'TEXT', 'token_count': 180}], 'total_token_count': 186, 'traffic_type': 'ON_DEMAND'}, 'invocation_id': 'e-2af5ee30-0171-4033-8655-90206d6199b6', 'author': 'weather_time_agent', 'actions': {'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}}, 'long_running_tool_ids': [], 'id': '48b0ce6a-4493-428d-8cb2-dd825dbbb80b', 'timestamp': 1757441607.458454}
Event: {'content': {'parts': [{'function_response': {'id': 'adk-411dfb07-a0fb-4357-b72c-652d7a86fff0', 'name': 'get_weather', 'response': {'status': 'success', 'report': 'The weather in New York is sunny with a tempe